In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json, os, glob, time, torch
from torch.optim import AdamW

# ⚡ [개선] pandarallel -> swifter로 교체 (더 안정적임)
import swifter 
from datasets import Dataset, DatasetDict, ClassLabel
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
    EarlyStoppingCallback
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
nltk.download('punkt', quiet=True)
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

SEED = 2025
np.random.seed(SEED)
plt.rc('font', family='NanumGothic')
plt.style.use('ggplot')
print("  - 딥러닝 라이브러리 (swifter 포함) 임포트 완료.")

In [ ]:
# [셀 3/7] 3. [분석 1] "정답지" (먼치_말뭉치) 로드 및 BLEU 분석
print("\n--- [3/7] 분석 1: '정답지' (먼치_말뭉치) 로드 및 BLEU 분석 ---")

munch_folder_path = './data/munch_data/'
csv_files = glob.glob(os.path.join(munch_folder_path, '*.csv'))
print(f"  - (1/3) '{munch_folder_path}'에서 {len(csv_files)}개의 원본 CSV 파일을 찾았습니다.")

munch_list = []
for f in csv_files:
    munch_list.append(pd.read_csv(f, header=None, names=['original', 'corrected']))
df_munch = pd.concat(munch_list, ignore_index=True)
print(f"  - (1/3) 'df_munch' (원본) 생성 완료. (Shape: {df_munch.shape})")

print("\n  - (2/3) 증강 데이터 로드 및 통합 중...")
try:
    aug_file_path = './data/augmented_by_hand_fixed_v2.txt'
    df_aug = pd.read_csv(aug_file_path, header=None, names=['original', 'corrected'])
    print(f"  - '{aug_file_path}'에서 증강 데이터 {df_aug.shape[0]}건 로드 완료.")
    
    df_munch = pd.concat([df_munch, df_aug], ignore_index=True)
    print(f"  - [최종] 원본 + 증강 데이터 통합 완료! (Total Shape: {df_munch.shape})")
    
except FileNotFoundError:
    print(f"  - ⚠️ [경고] 증강 데이터 파일 '{aug_file_path}'를 찾을 수 없습니다.")
    print("  - ⚠️ 원본 데이터로만 분석을 진행합니다.")
except Exception as e:
    print(f"  - ⚠️ [경고] 증강 데이터 로드 중 오류 발생: {e}")


print("\n  - (3/3) 'BLEU' 점수 계산 시작 (swifter 병렬 처리)...")

# (calculate_bleu 함수는 동일하게 유지)
def calculate_bleu(original, corrected):
    try:
        original_tokens = str(original).split()
        corrected_tokens = [str(corrected).split()]
        if not original_tokens or not corrected_tokens[0]:
            return 0.0
        chencherry = SmoothingFunction()
        score = sentence_bleu(corrected_tokens, original_tokens, smoothing_function=chencherry.method1)
        return score
    except ZeroDivisionError:
        print("  - [경고] calculate_bleu에서 ZeroDivisionError 발생. 0.0 반환.", flush=True)
        return 0.0
    except Exception as e:
        print(f"  - [경고] calculate_bleu에서 예외 발생: {e}. None 반환.", flush=True)
        return None

start_time = time.time()
# ⚡ [개선] df_munch.parallel_apply -> df_munch.swifter.apply로 변경
# (swifter는 .swifter 접근자를 사용합니다)
df_munch['bleu_score'] = df_munch.swifter.apply(lambda row: calculate_bleu(row['original'], row['corrected']), axis=1)
end_time = time.time()
print(f"  - 'BLEU' 점수 계산 완료. (소요 시간: {end_time - start_time:.2f} 초)")

print("\n  - [시각화 1] 'BLEU' 점수(난이도) 분포 그래프 생성 중...")
plt.figure(figsize=(8, 5))
sns.histplot(df_munch['bleu_score'], bins=50, kde=True)
plt.title('AI 교정 난이도 (BLEU) 분포 (증강 데이터 포함)', fontsize=16)
plt.xlabel('BLEU 점수 (0: 어려움, 1: 쉬움)')
plt.ylabel('문장 수', fontsize=12)
plt.savefig('./output/images/bleu_score_hist_augmented.png')
print("  - 'bleu_score_hist_augmented.png' 저장 완료.")
plt.show()

In [ ]:
# [셀 4/7] 4. [분석 2] "문장교정기록" 로드 및 정성적 분석
print("\n--- [4/7] 분석 2: '문장교정기록.json' 로드 및 정성적 분석 ---")
print("  - (이 셀은 모델링 파이프라인에 영향을 주지 않으므로 변경 없이 실행됩니다.)")

try:
    correction_file_path = './data/문장교정기록.json'
    df_corrections = pd.read_json(correction_file_path)

    df_corrections['candidate_1'] = df_corrections['output_sentences'].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)
    df_corrections['candidate_2'] = df_corrections['output_sentences'].apply(lambda x: x[1] if isinstance(x, list) and len(x) > 1 else None)
    df_corrections['candidate_3'] = df_corrections['output_sentences'].apply(lambda x: x[2] if isinstance(x, list) and len(x) > 2 else None)
    print(f"  - 'df_corrections' (Shape: {df_corrections.shape}) 로드 및 펼치기 완료.")

    print("\n" + "="*50)
    print("🕵️‍♂️ [정성적 분석] AI 수정 샘플 (샘플 10개)")
    print("="*50)
    pd.set_option('display.max_colwidth', 300)
    display(df_corrections[['input_sentence', 'candidate_1']].sample(10, random_state=SEED))
    pd.reset_option('display.max_colwidth')

except FileNotFoundError:
    print(f"  - ⚠️ 오류: '{correction_file_path}' 파일을 찾을 수 없습니다.")
except Exception as e:
    print(f"  - ⚠️ 분석 중 오류 발생: {e}")

In [ ]:
# [셀 5/7] 5. [모델링 1] 딥러닝용 "훈련/정답(X, y)" 데이터 생성
print("\n--- [5/7] [모델링 1] 딥러닝용 '훈련 데이터' 생성 ---")
print("  - 'BLEU 점수'를 기준으로 '쉬운 교정(0)'과 '어려운 교정(1)'을 분류합니다.")

try:
    df_train = df_munch[['original', 'bleu_score']].copy()

    def classify_difficulty(score):
        if score is None: return -1
        if score < 0.1: return 1 # (1: 어려움)
        elif score > 0.8: return 0 # (0: 쉬움)
        else: return -1 # (중간 제외)

    df_train['label'] = df_train['bleu_score'].apply(classify_difficulty)
    df_train = df_train[df_train['label'] != -1].copy()
    df_train = df_train.rename(columns={'original': 'text'})
    
    # --- ⚡ [수정] 데이터 정제 (NaN 결측치 제거 및 문자열 변환) ---
    print(f"  - [데이터 정제 전] Shape: {df_train.shape}")
    
    # 1. 'text' 컬럼에 NaN (결측치)이 있는 행을 제거 (토크나이저 오류 방지)
    df_train = df_train.dropna(subset=['text'])
    
    # 2. 'text' 컬럼의 모든 데이터를 '문자열(str)' 타입으로 강제 변환
    df_train['text'] = df_train['text'].astype(str)
    
    print(f"  - [데이터 정제 후] Shape: {df_train.shape}")
    # --- 여기까지 수정 ---
    
    print(f"\n  - '최종 훈련 데이터' (X, y) 분포:")
    print(df_train['label'].value_counts())

    # Pandas -> Hugging Face 'Dataset'으로 변환
    dataset = Dataset.from_pandas(df_train[['text', 'label']])

    # --- 'label' 컬럼을 'ClassLabel'로 "먼저" 변환합니다. ---
    dataset = dataset.cast_column("label", ClassLabel(num_classes=2, names=['easy', 'hard']))

    # "그 다음" train_test_split을 실행합니다.
    train_test_dataset = dataset.train_test_split(test_size=0.2, seed=SEED, stratify_by_column='label')
    dataset_dict = DatasetDict({
        'train': train_test_dataset['train'],
        'test': train_test_dataset['test']
    })
    print("\n  - 'Hugging Face Dataset' 생성 완료:")
    print(dataset_dict)

except NameError:
     print("  - ⚠️ 'df_munch'가 로드되지 않아 훈련 데이터 생성을 건너뜁니다.")
except Exception as e:
    print(f"  - ⚠️ 훈련 데이터 생성 중 오류 발생: {e}")

In [ ]:
# [셀 6/7] 6. [모델링 2] 딥러닝 모델 및 토크나이저 로드 (PyTorch)
print("\n--- [6/7] [모델링 2] 딥러닝 모델 로드 (PyTorch) ---")

try:
    # ⚡ [개선] VRAM 10GB 제약 하에 'klue/roberta-base'로 변경 (User Item 3)
    # (대안: "kykim/bert-kor-base")
    MODEL_NAME = "kykim/bert-kor-base"
    print(f"  - 딥러닝 모델: {MODEL_NAME} (VRAM 10GB 최적화 모델)")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    # ⚡ [개선] 'Dynamic Padding'을 위해 'padding' 옵션 제거 (User Item 5)
    def tokenize_function(examples):
        # (VRAM 낭비와 속도 저하의 주범인 padding="max_length" 제거!)
        # (DataCollator가 배치 단위로 필요한 만큼만 동적 패딩)
        return tokenizer(examples['text'], truncation=True, max_length=128)

    tokenized_datasets = dataset_dict.map(tokenize_function, batched=True, remove_columns=['text'])
    print("  - '토크나이징' (숫자 변환) 완료. (Dynamic Padding 준비됨)")

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2) # (0: 쉬움, 1: 어려움)

    print(f"\n  - [활성화 함수 확인]: {model.config.hidden_act}")
    print("  - 모델 로드 완료. 훈련 준비가 끝났습니다.")

except NameError:
    print("  - ⚠️ 'dataset_dict'가 정의되지 않아 모델 로드를 건너뜁니다.")
except Exception as e:
    print(f"  - ⚠️ 모델 로드 중 오류 발생: {e}")

In [ ]:
# [셀 7/7] 7. [모델링 3] 모델 훈련 (PyTorch + AdamW) 및 평가
print("\n--- [7/7] [모델링 3] '딥러닝 모델' 훈련 및 평가 ---")
print("  - (GPU(A100/10GB) 사용, Dynamic Padding 및 Early Stopping 적용)")

try:
    # 1. 훈련 설정 (Training Arguments) 정의
    training_args = TrainingArguments(
        output_dir="./output/model_checkpoints",

        # ⚡ [개선] 훈련 전략 변경 (증강 데이터셋 반영 및 Early Stopping)
        num_train_epochs=5, 
        
        # ⚡ [수정] 'evaluation_strategy' -> 'eval_strategy'로 변경 (구형 버전 호환)
        eval_strategy="epoch",          # 1 에포크마다 평가
        
        # ⚡ [수정] save_strategy를 eval_strategy와 일치시킴 (오류 해결)
        save_strategy="epoch",          # 1 에포크마다 저장
        
        logging_strategy="epoch",       # 1 에포크마다 로깅
        load_best_model_at_end=True,    # (필수) 훈련 종료 시 최고 성능 모델 로드
        metric_for_best_model="eval_f1", # (불균형 데이터이므로 'f1' 기준)

        per_device_train_batch_size=32, 
        per_device_eval_batch_size=32,
        seed=SEED,
        report_to="none" 
    )

    # 2. 평가 지표 정의 (Accuracy, F1-Score)
    def compute_metrics(eval_pred):
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)
        return {
            'accuracy': accuracy_score(labels, predictions),
            'f1': f1_score(labels, predictions, average='macro') 
        }

    # 3. 'DataCollator' 정의 (Dynamic Padding)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # 4. '트레이너' 정의
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["test"],
        tokenizer=tokenizer,
        compute_metrics=compute_metrics,
        data_collator=data_collator,
        
        # ⚡ [개선] EarlyStopping 콜백 추가 (patience=3)
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
    )

    # 5. 훈련 시작!
    trainer.train()
    print("\n  - 훈련 완료!")

    # 6. 평가 (load_best_model_at_end=True에 의해 "최고 성능" 모델로 평가됨)
    print("\n" + "="*50)
    print("🏆 [딥러닝 모델 최종 성능 평가 (Best Model)]")
    print("="*50)
    eval_results = trainer.evaluate()
    print(eval_results)

    print("\n" + "="*50)
    print("💡 [최종 제안] 스마트 라우터 (Smart Router)")
    print("="*50)
    print(f"  - 이 모델(Macro F1: {eval_results['eval_f1']*100:.0f}%)을 (주)북엔드 서버에 탑재하여,")
    print(f"  - '어려움(1)'으로 예측된 문장은 'gpt-4.1'로 보내고,")
    print(f"  - '쉬움(0)'으로 예측된 문장은 'gemini-flash'로 보내는,")
    print(f"  - '초정밀 스마트 라우터'를 완성할 수 있습니다!")

except NameError:
    print("  - ⚠️ 'model' 또는 'tokenized_datasets'가 정의되지 않아 훈련을 건너뜁니다.")
except torch.cuda.OutOfMemoryError:
    print("  - ⚠️ GPU 메모리 부족 오류! (OOM)")
    print("  - [!] per_device_train_batch_size를 32 -> 16으로 줄이고, '커널 재시작' 후 다시 시도하세요.")
except Exception as e:
    print(f"  - ⚠️ 훈련 중 오류 발생: {e}")

In [ ]:
# [셀 8/8] 8. [모델링 4] "경영진 친화적" 혼동 행렬 시각화
print("\n--- [8/8] [모델링 4] '딥러닝 모델' 최종 성능 시각화 ---")
print("  - '경영진'과 '개발자'의 관점을 모두 담은 '혼동 행렬'을 생성합니다.")

try:
    # --- 1. [데이터 준비] '테스트 데이터셋'에서 '예측'과 '정답'을 가져옵니다 ---
    print("  - 테스트 데이터로 '예측'을 다시 수행합니다...")
    # 'trainer' (훈련된 모델)를 사용해 'test' 데이터셋의 예측 결과를 가져옵니다.
    predictions = trainer.predict(tokenized_datasets["test"])

    # 'predictions.predictions'는 '숫자'(Logits)이므로, 'argmax'로 '0' 또는 '1'로 변환합니다.
    y_pred = np.argmax(predictions.predictions, axis=1)
    # 'predictions.label_ids'가 '진짜 정답(y_test)'입니다.
    y_test = predictions.label_ids

    # ⚡ [개선] 그래프 제목에 '모델명' 자동 반영
    model_name = MODEL_NAME.split('/')[-1] # (e.g., "klue-roberta-base")

    # --- 2. '경영진 친화적' 혼동 행렬 생성 ---
    print(f"  - '{model_name}' 모델의 '경영진 친화적' 혼동 행렬 생성 중...")
    cm = confusion_matrix(y_test, y_pred)

    # [중요!] 'label'의 의미 (0: 쉬움, 1: 어려움)
    # cm[0, 0] = TN (정답:0, 예측:0) - '쉬움'을 '쉽다'고 맞힘 (비용 절감!)
    # cm[0, 1] = FP (정답:0, 예측:1) - '쉬움'을 '어렵다'고 실수 (불필요한 비용!)
    # cm[1, 0] = FN (정답:1, 예측:0) - '어려움'을 '쉽다'고 실수 (치명적 실수! 🚨)
    # cm[1, 1] = TP (정답:1, 예측:1) - '어려움'을 '어렵다'고 맞힘 (만족도 향상!)

    # '경영진'이 보기 편하도록 [TP, FN], [FP, TN] 순서로 '재배열'합니다.
    # (여기서 1='어려움'(Positive), 0='쉬움'(Negative)입니다.)
    reordered_cm = np.array([[cm[1, 1], cm[1, 0]],   # [TP, FN]
                             [cm[0, 1], cm[0, 0]]])  # [FP, TN]

    names = ["True Positive \n(어려움을 '어렵다'고 예측)",
             "False Negative \n(어려움을 '쉽다'고 실수!)",
             "False Positive \n(쉬움을 '어렵다'고 실수)",
             "True Negative \n(쉬움을 '쉽다'고 예측)"]

    counts = [value for value in reordered_cm.flatten()]
    percentages = ["{0:.2%}".format(value / np.sum(reordered_cm)) for value in reordered_cm.flatten()]
    labels = [f"{v1}\n{v2}\n{v3}" for v1, v2, v3 in zip(names, counts, percentages)]
    labels = np.asarray(labels).reshape(2, 2)

    plt.figure(figsize=(6, 4))

    sns.heatmap(reordered_cm, annot=labels, cmap="Blues", fmt="", annot_kws={"size": 12}) # 폰트 크기 조절
    plt.title(f"스마트 라우터 엔진 성능 ({model_name})", fontsize=16)
    plt.xticks([0.5, 1.5], ["예측: 1 (어려움)", "예측: 0 (쉬움)"])
    plt.yticks([0.5, 1.5], ["정답: 1 (어려움)", "정답: 0 (쉬움)"], rotation=0)
    plt.xlabel('AI의 예측 (라우팅 경로)', fontsize=12)
    plt.ylabel('실제 정답 (난이도)', fontsize=12)
    plt.savefig(f"./output/images/CM_{model_name}.png")
    print(f"  - 'CM_{model_name}.png' 저장 완료.")
    plt.show()

except NameError:
    print("  - ⚠️ 'trainer' 또는 'tokenized_datasets'가 정의되지 않아 시각화를 건너뜁니다.")
    print("  - [!] [셀 7/7]의 '훈련'이 먼저 성공했는지 확인하세요!")
except Exception as e:
    print(f"  - ⚠️ 시각화 중 오류 발생: {e}")

> 🤖 텍스트 난이도 분류 모델(정확도 98%)을 활용해,  
> 어려운 문장은 gpt-4.1으로, 쉬운 문장은 gemini-flash로 자동 분배하는 **‘초정밀 스마트 라우터’**를 구축합니다.  
> 이를 통해 **비용과 만족도를 동시에 최적화**할 수 있습니다.